# Retrain U-Net Decoder on Shared Encoder

**Goal:** Take the encoder from `classifier.pth` (best-performing encoder),
freeze it, and train only the UNet decoder + bottleneck on top.
This produces a `unet.pth` whose encoder matches the classifier/localizer,
so the multi-task model can use a single shared encoder.

**BEFORE RUNNING:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **ON**
3. Click **Save Version** → **Save & Run All (Commit)**

In [ ]:
import os
os.environ["WANDB_API_KEY"]      = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT = "/kaggle/working/checkpoints"
os.makedirs(CKPT, exist_ok=True)

!git clone https://github.com/usnaveen/A2_Deep_Learning.git /kaggle/working/repo 2>&1 | tail -3
%cd /kaggle/working/repo
!pip install -q wandb albumentations gdown scikit-learn

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
assert torch.cuda.is_available()

In [ ]:
import gdown

# Download the classifier checkpoint (has the encoder we want to use)
CLASSIFIER_ID = "1Ob4ecaMOaoH-56TdIQ9lYEdGKfn1jy8L"
cls_path = f"{CKPT}/classifier.pth"
print("Downloading classifier.pth (for encoder)...")
gdown.download(id=CLASSIFIER_ID, output=cls_path, quiet=True, fuzzy=True)
print(f"  OK: {os.path.getsize(cls_path)/1e6:.0f} MB")

In [ ]:
import re, torch

def _canonicalize_checkpoint(sd):
    """Normalize nested-block key format to our flat convention."""
    if not any(re.match(r'encoder\.block\d+\.\d+\.\d+\.', k) for k in sd):
        return sd
    new_sd = {}
    for k, v in sd.items():
        m = re.match(r'encoder\.block(\d+)\.(\d+)\.(\d+)\.(.*)', k)
        if m:
            N, M, L, suf = int(m.group(1)), int(m.group(2)), int(m.group(3)), m.group(4)
            X = N - 1
            if X in (0, 1):
                Y = 0 if L == 0 else 1
            else:
                Y = (0 if L == 0 else 1) if M == 0 else (3 if L == 0 else 4)
            new_sd[f'encoder.block{X}.{Y}.{suf}'] = v
        elif k.startswith('classifier.'):
            new_sd['head.' + k[len('classifier.'):]] = v
        elif k.startswith('regressor.'):
            new_sd['head.' + k[len('regressor.'):]] = v
    return new_sd

# Load and remap classifier checkpoint
cls_state = _canonicalize_checkpoint(
    torch.load(cls_path, map_location='cpu', weights_only=False)
)

# Extract just the encoder weights
encoder_weights = {k[len('encoder.'):]: v for k, v in cls_state.items()
                   if k.startswith('encoder.')}
print(f"Encoder: {len(encoder_weights)} keys extracted from classifier")

In [ ]:
from models.segmentation import VGG11UNet

device = torch.device('cuda')
model = VGG11UNet(num_classes=3).to(device)

# Load the classifier's encoder into the UNet's encoder
missing, unexpected = model.encoder.load_state_dict(encoder_weights, strict=False)
print(f"Encoder loaded: {len(missing)} missing, {len(unexpected)} unexpected")

# FREEZE the entire encoder — we only train decoder + bottleneck
for param in model.encoder.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable: {trainable/1e6:.1f}M  |  Frozen: {frozen/1e6:.1f}M")

In [ ]:
from data.pets_dataset import download_oxford_pet, create_dataloaders

download_oxford_pet('./data/oxford_pet')
train_loader, val_loader, _ = create_dataloaders(
    root='./data/oxford_pet', batch_size=16, num_workers=2
)
print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

In [ ]:
import time
import wandb
import numpy as np
import torch.nn as nn
from train import compute_dice_score

EPOCHS = 20
LR = 1e-3

wandb.init(project='da6401-assignment2',
           name='unet_shared_encoder_20ep',
           tags=['segmentation', 'shared-encoder'])

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([1.0, 1.0, 2.0]).to(device))
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6)

best_dice = 0.0
save_path = f"{CKPT}/unet.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    train_loss = 0.0
    for batch in train_loader:
        imgs  = batch['image'].to(device)
        masks = batch['mask'].to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    # Validation
    model.eval()
    dice_sum, n = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            pred = model(batch['image'].to(device)).argmax(1).cpu()
            bs = pred.size(0)
            dice_sum += compute_dice_score(pred, batch['mask']) * bs
            n += bs
    val_dice = dice_sum / n
    elapsed = time.time() - t0

    wandb.log({'epoch': epoch,
               'train/loss': train_loss / len(train_loader),
               'val/dice': val_dice,
               'lr': scheduler.get_last_lr()[0]})

    tag = ''
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), save_path)
        tag = '  ** saved'
    print(f"  Epoch {epoch:2d}/{EPOCHS} | "
          f"Loss {train_loss/len(train_loader):.4f} | "
          f"Val Dice {val_dice:.4f} | "
          f"{elapsed:.0f}s{tag}")

wandb.finish()
print(f"\nBest Val Dice: {best_dice:.4f}")
print(f"Saved to: {save_path}")

## Verify: load as multi-task model (single encoder)

In [ ]:
from sklearn.metrics import f1_score
from models.classification import VGG11Classifier
from models.localization import VGG11Localizer
from models.segmentation import VGG11UNet
from train import compute_iou, compute_dice_score

print("="*55)
print("  SINGLE-ENCODER VERIFICATION")
print("="*55)

# The shared encoder comes from the classifier checkpoint
encoder_sd = encoder_weights  # already extracted above

# ---- Classification ----
cls_model = VGG11Classifier(num_classes=37, dropout_p=0.5, use_bn=True).to(device)
cls_full = _canonicalize_checkpoint(
    torch.load(cls_path, map_location=device, weights_only=False))
cls_model.load_state_dict(
    {k: v for k, v in cls_full.items()
     if k.startswith('encoder.') or k.startswith('head.')},
    strict=False)
cls_model.eval()
preds, labels = [], []
with torch.no_grad():
    for b in val_loader:
        preds.extend(cls_model(b['image'].to(device)).argmax(1).cpu().tolist())
        labels.extend(b['label'].tolist())
f1 = f1_score(labels, preds, average='macro')
print(f"  Classification  F1   = {f1:.4f}  {'OK' if f1>=0.80 else 'LOW'}")

# ---- Segmentation (new UNet with shared encoder) ----
unet = VGG11UNet(num_classes=3).to(device)
unet.load_state_dict(
    torch.load(save_path, map_location=device, weights_only=False),
    strict=False)
unet.eval()
dice_sum, n = 0.0, 0
with torch.no_grad():
    for b in val_loader:
        pred = unet(b['image'].to(device)).argmax(1).cpu()
        dice_sum += compute_dice_score(pred, b['mask']) * b['image'].size(0)
        n += b['image'].size(0)
dice = dice_sum / n
print(f"  Segmentation    Dice = {dice:.4f}  {'OK' if dice>=0.30 else 'LOW'}")

# ---- Verify encoder match ----
# Check that the unet's encoder matches the classifier's encoder
match = all(
    torch.equal(
        dict(unet.encoder.named_parameters())[k],
        dict(cls_model.encoder.named_parameters())[k])
    for k in dict(cls_model.encoder.named_parameters())
)
print(f"  Encoder match   = {match}  (should be True)")
print("="*55)

In [ ]:
print("="*55)
print("  OUTPUT FILES — download from Output tab")
print("="*55)
for f in sorted(os.listdir(CKPT)):
    mb = os.path.getsize(f"{CKPT}/{f}") / 1e6
    print(f"  {f:30s}  {mb:.0f} MB")
print()
print("  Next steps:")
print("  1. Download checkpoints/unet.pth from Output tab")
print("  2. Upload to Google Drive, get share link")
print("  3. Update UNET_DRIVE_ID in models/multitask.py")
print("  4. Revert multitask.py to single encoder")
print("  5. Rebuild ZIP and resubmit to Gradescope")